In [1]:
!pip  install bcrypt

In [4]:
import bcrypt

In [22]:
# url="http://localhost:3000/projects"
url="http://localhost:3000/login"
headers ={ "Content-Type": "application/x-www-form-urlencoded" }
import requests
headers={}
data={"user_login":"admin",
     'user_password': "1234567890"}
r=requests.post(url,headers=headers,data=data)
cookie=r.cookies

In [ ]:
r=requests(url,coo)

In [27]:
# bcrypt.hashpw(b'password',b'pass')
! pip install bs4

In [41]:
import requests
from bs4 import *
url="http://localhost:3000/login"
r=requests.get(url)
"""Extract CSRF token from HTML"""
html_content=r.text
soup = BeautifulSoup(html_content, 'html.parser')
# token = soup.find('input', {'name': 'authenticity_token'})
# return token['content'] if token else None
form=soup.find('form')

In [49]:
soup.find('input', {'name': 'authenticity_token'}).attrs.get('value')

'qJYQ+edYK16D6TTl68IzNIL9EM1UghCuRzdxqbLq1uC3sWX8Hq9FR4p+UBBXnc6knLG0KfGGwLxm52oIlfvcrw=='

In [75]:
from bs4 import BeautifulSoup
import requests

# Получаем HTML страницы
response = requests.get('http://localhost:3000/login')
soup = BeautifulSoup(response.text, 'html.parser')

# Извлекаем токен
token_input = soup.find('input', {'name': 'authenticity_token'}).attrs
# token_input.get('value')
# if token_input:
authenticity_token = token_input.get('value')

    
"""Get list of projects"""
response = self.session.get(f"{self.base_url}/projects")
soup = BeautifulSoup(response.text, 'html.parser')

projects = []
project_elements = soup.select('#list-active-projects .project_description a')

for project in project_elements:
    projects.append({
        'name': project.text.strip(),
        'url': project['href']
    })
# mess="CSRF Token: {}".format(authenticity_token) 
# print(authenticity_token)
# type(authenticity_token)
# else:
# print("Token input not found")

# # # Или через CSS-селектор
# # token = soup.select_one('meta[name="csrf-token"]')['content']
# # print(f"Alternative CSRF Token: {token}")

NameError: name 'self' is not defined

In [1]:
import requests
from bs4 import BeautifulSoup

class TracksClient:
    def __init__(self, base_url="http://localhost:3000"):
        self.base_url = base_url
        self.session = requests.Session()
        self._authenticity_token = None

    def _get_authenticity_token(self, url=None):
        """Получает CSRF токен со страницы"""
        url = url or f"{self.base_url}/login"
        response = self.session.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Ищем токен в meta-теге или форме
        token = soup.find('meta', {'name': 'csrf-token'}) or \
                soup.find('input', {'name': 'authenticity_token'})
        
        if token:
            self._authenticity_token = token.get('content') or token.get('value')
        return self._authenticity_token

    def login(self, username, password):
        """Аутентификация в системе (FR-003)"""
        login_url = f"{self.base_url}/login"
        
        # Получаем токен
        self._get_authenticity_token(login_url)
        if not self._authenticity_token:
            raise Exception("CSRF token not found")

        # Формируем данные формы
        form_data = {
            'utf8': '✓',
            'authenticity_token': self._authenticity_token,
            'user_login': username,
            'user_password': password,
            'commit': 'Login'
        }

        # Отправляем запрос
        response = self.session.post(login_url, data=form_data)
        
        # Проверяем успешность входа
        if "Invalid username or password" in response.text:
            raise Exception("Authentication failed")
        
        return response.status_code == 200